# Syria DroughtLens
**FTL Syria AI4Climate: Python for Climate Hackathon**

**Climate challenge:** Water and drought

**Research question:** Have drought conditions in Syria become more frequent or severe between 1981 and 2025, and which regions are most affected?

**Data:** NASA POWER Daily API (MERRA-2 reanalysis): rainfall (`PRECTOTCORR`, mm/day), temperature (`T2M`, C), plus soil wetness and other variables, 1981-2025, 8 Syrian locations.
https://power.larc.nasa.gov/docs/services/api/temporal/daily/

**How to run:** *Runtime > Run all*. The first run downloads the data (a few minutes) and caches it in `data/raw/`.

## Definitions used throughout
- **Hydrological year:** October to September, named after the year it ends in (Oct 1981 - Sep 1982 = 1982). Only complete years 1982-2025 (44 years) are used.
- **Rainfall anomaly (z):** annual rainfall standardised per region: `(x - mean) / std`.
- **Dry year:** z <= -0.5. **Drought year:** z <= -1.
- **Hot year:** temperature z >= 0.5. **Hot-dry year:** dry *and* hot.
- **Periods:** 1982-2003 vs 2004-2025 (22 years each).

## 1. Setup: configuration and functions

In [ ]:
"""Project configuration for Syria DroughtLens (single source of truth).

Every location, variable name, threshold and file path lives here, so that
changing a setting once changes it everywhere.
"""
from pathlib import Path

# ---- Study locations: name -> (latitude, longitude) ----------------------
# NASA POWER uses a ~0.5 x 0.625 degree grid, so nearby points can share a
# cell (e.g. Damascus city and its countryside). prepare.check_raw() verifies
# that no two locations returned identical rainfall.
LOCATIONS = {
    "Damascus":    (33.51, 36.29),
    "Aleppo":      (36.20, 37.16),
    "Homs":        (34.73, 36.72),
    "Latakia":     (35.52, 35.79),
    "Raqqa":       (35.95, 39.01),
    "Deir ez-Zor": (35.34, 40.14),
    "Hasakah":     (36.50, 40.75),
    "Daraa":       (32.63, 36.10),
}

# ---- NASA POWER request ---------------------------------------------------
POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
COMMUNITY = "AG"
CORE_PARAMS = ["PRECTOTCORR", "T2M"]          # rainfall (mm/day), mean temp (C)
EXTRA_PARAMS = ["T2M_MAX", "T2M_MIN", "RH2M", "GWETROOT", "GWETTOP", "EVLAND"]
START_DATE, END_DATE = "19810101", "20251231"
MISSING_VALUE = -999                          # NASA POWER's missing-data flag

# ---- Analysis settings ----------------------------------------------------
# Hydrological year runs Oct-Sep and is named after the year it ends in
# (Oct 1981 - Sep 1982 = 1982). Only complete years 1982-2025 are used.
FIRST_HYDRO_YEAR, LAST_HYDRO_YEAR = 1982, 2025
SPLIT_YEAR = 2003                             # last year of the first period
PERIODS = ("1982-2003", "2004-2025")          # two equal halves, 22 years each
WET_SEASON_MONTHS = (11, 12, 1, 2, 3)         # used for dry-spell length
DRY_DAY_MM = 1.0                              # a day below this is a "dry day"
DRY_Z = -0.5                                  # dry year:     rainfall z <= -0.5
DROUGHT_Z = -1.0                              # drought year: rainfall z <= -1.0
HOT_Z = 0.5                                   # hot year:     temperature z >= 0.5
SIGNIFICANCE = 0.05

# ---- Paths (relative to the folder you run from) --------------------------
DATA_RAW = Path("data/raw/power_raw.csv")
DATA_CLEAN = Path("data/processed/power_clean.csv")
DATA_ANNUAL = Path("data/processed/annual.csv")
TABLES_DIR = Path("outputs/tables")
FIG_DIR = Path("outputs/figures")

In [ ]:
"""Step 1: download daily climate data from the NASA POWER API."""
import time

import pandas as pd
import requests



def fetch_point(name, lat, lon, params, retries=3):
    """Download daily data for one location. Returns a DataFrame or None."""
    query = {
        "parameters": ",".join(params),
        "community": COMMUNITY,
        "latitude": lat,
        "longitude": lon,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON",
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(POWER_URL, params=query, timeout=180)
            response.raise_for_status()
            data = response.json()["properties"]["parameter"]
            df = pd.DataFrame(data)
            df.index = pd.to_datetime(df.index, format="%Y%m%d")
            df.index.name = "date"
            df.insert(0, "region", name)
            return df
        except Exception as exc:
            print(f"  {name}: attempt {attempt}/{retries} failed -> {exc}")
            time.sleep(3)
    return None


def fetch_all(locations=None):
    """Download every location and stack them into one DataFrame."""
    locations = locations or LOCATIONS
    frames = []
    for name, (lat, lon) in locations.items():
        print(f"Fetching {name}...")
        df = fetch_point(name, lat, lon, CORE_PARAMS + EXTRA_PARAMS)
        if df is None:
            print(f"  {name}: retrying with core variables only")
            df = fetch_point(name, lat, lon, CORE_PARAMS)
        if df is None:
            # Fail loudly: silently skipping a region would bias every result.
            raise RuntimeError(f"Could not download data for {name}")
        frames.append(df)
        time.sleep(1)  # be polite to the API
    return pd.concat(frames)

In [ ]:
"""Step 2: load, validate and clean the raw data."""
import numpy as np
import pandas as pd



def load_raw(path):
    """Read the raw CSV written by fetch_all()."""
    return pd.read_csv(path, parse_dates=["date"], index_col="date")


def check_raw(raw):
    """Sanity-check the raw data. Raises an error on serious problems."""
    counts = raw.groupby("region").size()
    print("Rows per region:")
    print(counts.to_string())
    if counts.nunique() != 1:
        raise ValueError("Regions have different numbers of rows")

    for region, d in raw.groupby("region"):
        if not d.index.is_unique:
            raise ValueError(f"{region}: duplicate dates")
        if not d.index.is_monotonic_increasing:
            raise ValueError(f"{region}: dates are not sorted")

    # Two points in the same grid cell return identical data and would be
    # double-counted as separate regions.
    rain = raw.reset_index().pivot(index="date", columns="region", values="PRECTOTCORR")
    names = list(rain.columns)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if np.allclose(rain[a], rain[b], equal_nan=True):
                raise ValueError(f"{a} and {b} have identical rainfall (same grid cell). Remove one.")

    numeric = raw.drop(columns="region")
    print("\nMissing-value flags (-999) per column:")
    print((numeric == MISSING_VALUE).sum().to_string())
    print("\nEmpty (NaN) cells per column:")
    print(numeric.isna().sum().to_string())


def clean(raw):
    """Clean the raw data and add time features.

    - -999 becomes NaN.
    - Short gaps (up to 3 days) are interpolated for temperature-type
      variables only. Rainfall is never interpolated: it is not a smooth signal.
    - Adds year, month and hydrological year (Oct-Sep, named by the end year).
    - Keeps only complete hydrological years (1982-2025).
    """
    df = raw.replace(MISSING_VALUE, np.nan)
    numeric_cols = [c for c in df.columns if c != "region"]
    fill_cols = [c for c in numeric_cols if c != "PRECTOTCORR"]
    df[fill_cols] = df.groupby("region")[fill_cols].transform(lambda s: s.interpolate(limit=3))

    df["year"] = df.index.year
    df["month"] = df.index.month
    df["hydro_year"] = np.where(df["month"] >= 10, df["year"] + 1, df["year"])
    df = df[(df["hydro_year"] >= FIRST_HYDRO_YEAR) & (df["hydro_year"] <= LAST_HYDRO_YEAR)].copy()

    remaining = df[numeric_cols].isna().sum()
    if remaining.any():
        print("WARNING: missing values remain after cleaning:")
        print(remaining[remaining > 0].to_string())
    return df

In [ ]:
"""Step 3: annual indicators, drought detection, trends and regional comparison."""
import numpy as np
import pandas as pd
from scipy import stats



def longest_run(mask):
    """Length of the longest run of consecutive True values."""
    best = current = 0
    for value in mask:
        current = current + 1 if value else 0
        best = max(best, current)
    return best


def _zscore(series):
    return (series - series.mean()) / series.std()


def build_annual(df):
    """One row per region and hydrological year."""
    agg = {"rain_mm": ("PRECTOTCORR", "sum"), "temp_mean": ("T2M", "mean")}
    optional = {  # only used if the extra variables were downloaded
        "tmax_mean": ("T2M_MAX", "mean"),
        "soil_root": ("GWETROOT", "mean"),
        "evap": ("EVLAND", "mean"),
    }
    for name, (column, func) in optional.items():
        if column in df.columns:
            agg[name] = (column, func)
    annual = df.groupby(["region", "hydro_year"]).agg(**agg).reset_index()

    # Longest run of dry days (<1 mm) inside the wet season (Nov-Mar).
    # Summer is dry every year, so a full-year dry spell would say nothing.
    wet = df[df["month"].isin(WET_SEASON_MONTHS)]
    spell = (wet.groupby(["region", "hydro_year"])["PRECTOTCORR"]
                .apply(lambda s: longest_run((s < DRY_DAY_MM).values))
                .rename("wet_dry_spell")
                .reset_index())
    return annual.merge(spell, on=["region", "hydro_year"])


def add_flags(annual):
    """Add standardised anomalies, period label and drought flags.

    z-scores use each region's own 1982-2025 mean and standard deviation.
    Because every region warmed, temperature z-scores are higher in the
    second period by construction. That is intended (dry years now happen on
    a warmer baseline) but it must not be read as extra dry years.
    """
    a = annual.copy()
    a["rain_z"] = a.groupby("region")["rain_mm"].transform(_zscore)
    a["temp_z"] = a.groupby("region")["temp_mean"].transform(_zscore)
    a["period"] = np.where(a["hydro_year"] <= SPLIT_YEAR, PERIODS[0], PERIODS[1])
    a["dry"] = a["rain_z"] <= DRY_Z
    a["drought"] = a["rain_z"] <= DROUGHT_Z
    a["hot_dry"] = a["dry"] & (a["temp_z"] >= HOT_Z)
    return a


def trend_table(annual, variables=("rain_mm", "temp_mean", "tmax_mean", "soil_root")):
    """Linear trend per region: slope per decade and p-value.

    Note: 44 points per region and simple OLS. Year-to-year autocorrelation
    is ignored, and 8 regions x several variables means several tests, so
    treat p-values near 0.05 as borderline.
    """
    rows = []
    for region, d in annual.groupby("region"):
        for var in variables:
            if var not in d.columns:
                continue
            res = stats.linregress(d["hydro_year"], d[var])
            rows.append({"region": region, "variable": var,
                         "slope_per_decade": res.slope * 10, "p_value": res.pvalue})
    out = pd.DataFrame(rows)
    out["significant"] = out["p_value"] < SIGNIFICANCE
    return out


def period_comparison(annual):
    """First half vs second half: means, counts and changes."""
    first, second = PERIODS
    g = (annual.groupby(["region", "period"])
               .agg(rain=("rain_mm", "mean"), temp=("temp_mean", "mean"),
                    dry_years=("dry", "sum"), droughts=("drought", "sum"),
                    hot_dry_years=("hot_dry", "sum"))
               .unstack("period"))
    out = pd.DataFrame(index=g.index)
    for col in g.columns.levels[0]:
        out[f"{col}_{first}"] = g[(col, first)]
        out[f"{col}_{second}"] = g[(col, second)]
    out["rain_change_pct"] = (out[f"rain_{second}"] / out[f"rain_{first}"] - 1) * 100
    out["temp_change_C"] = out[f"temp_{second}"] - out[f"temp_{first}"]
    return out


def correlations(annual):
    """Correlation between annual rainfall and temperature / soil wetness."""
    rows = []
    for region, d in annual.groupby("region"):
        row = {"region": region, "rain_vs_temp": d["rain_mm"].corr(d["temp_mean"])}
        if "soil_root" in d.columns:
            row["rain_vs_soil"] = d["rain_mm"].corr(d["soil_root"])
        rows.append(row)
    return pd.DataFrame(rows).set_index("region")


def regional_ranking(annual):
    """Rank regions by recent drought frequency, with rainfall variability."""
    second = PERIODS[1]
    g = annual.groupby("region")
    out = pd.DataFrame({
        "mean_rain_mm": g["rain_mm"].mean(),
        "rain_cv": g["rain_mm"].std() / g["rain_mm"].mean(),
        "drought_years": g["drought"].sum(),
        "recent_droughts": annual[annual["period"] == second].groupby("region")["drought"].sum(),
        "hot_dry_years": g["hot_dry"].sum(),
    })
    return out.sort_values("recent_droughts", ascending=False)


def hot_dry_share(annual):
    """Percentage of dry years that were also hot, per region and period.

    Returns (share, dry_counts, hot_dry_counts). The last row of `share`
    pools all regions.
    """
    dry = annual.groupby(["region", "period"])["dry"].sum().unstack("period")
    hot_dry = annual.groupby(["region", "period"])["hot_dry"].sum().unstack("period")
    share = hot_dry / dry * 100
    share.loc["ALL REGIONS"] = hot_dry.sum() / dry.sum() * 100
    return share, dry, hot_dry


def hot_dry_test(annual):
    """Fisher exact test: is a dry year more likely to be hot in period 2?

    Caveat: regions share droughts (one dry year often hits several regions),
    so pooled observations are not independent. The p-value is indicative.
    """
    first, second = PERIODS

    def counts(period):
        d = annual[(annual["period"] == period) & annual["dry"]]
        return int(d["hot_dry"].sum()), int((~d["hot_dry"]).sum())

    hot2, not2 = counts(second)
    hot1, not1 = counts(first)
    odds, p = stats.fisher_exact([[hot2, not2], [hot1, not1]])
    return {f"hot_dry_{first}": hot1, f"dry_{first}": hot1 + not1,
            f"hot_dry_{second}": hot2, f"dry_{second}": hot2 + not2,
            "odds_ratio": odds, "p_value": p}

In [ ]:
"""Step 4: figures. Each function saves a PNG and can optionally display it."""
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats


sns.set_theme(style="whitegrid")


def region_order(annual):
    """Regions from wettest to driest (used for consistent ordering)."""
    return annual.groupby("region")["rain_mm"].mean().sort_values(ascending=False).index.tolist()


def _grid(n, ncols=4, **kwargs):
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, **kwargs)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes[:n]


def _finish(fig, filename, show):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / filename, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


def fig_anomaly_timeline(annual, show=True):
    """Rainfall anomaly per year and region; drought years in red."""
    order = region_order(annual)
    fig, axes = _grid(len(order), figsize=(18, 7), sharey=True)
    for ax, region in zip(axes, order):
        d = annual[annual["region"] == region]
        colors = np.where(d["rain_z"] <= DROUGHT_Z, "#c0392b",
                          np.where(d["rain_z"] < 0, "#e6b0aa", "#5dade2"))
        ax.bar(d["hydro_year"], d["rain_z"], color=colors)
        ax.axhline(DROUGHT_Z, ls="--", lw=1, color="gray")
        ax.set_title(region)
    fig.suptitle(f"Rainfall anomaly by hydrological year (red = drought, z <= {DROUGHT_Z})", fontsize=14)
    fig.supylabel("Standardised rainfall anomaly (z)")
    fig.tight_layout()
    _finish(fig, "fig1_anomaly_timeline.png", show)


def fig_drought_heatmap(annual, show=True):
    """Region x year heatmap of rainfall anomaly."""
    order = region_order(annual)
    pivot = annual.pivot(index="region", columns="hydro_year", values="rain_z").loc[order]
    fig, ax = plt.subplots(figsize=(16, 4.5))
    sns.heatmap(pivot, cmap="RdBu", center=0, vmin=-2.5, vmax=2.5,
                cbar_kws={"label": "Rainfall anomaly (z)"}, ax=ax)
    ax.set_title("Syria DroughtLens: rainfall anomaly by region and year")
    ax.set_xlabel("Hydrological year")
    ax.set_ylabel("")
    fig.tight_layout()
    _finish(fig, "fig2_drought_heatmap.png", show)


def fig_temperature_trends(annual, show=True):
    """Annual mean temperature with a linear trend line per region."""
    order = region_order(annual)
    fig, axes = _grid(len(order), figsize=(18, 7), sharey=True)
    for ax, region in zip(axes, order):
        d = annual[annual["region"] == region]
        res = stats.linregress(d["hydro_year"], d["temp_mean"])
        ax.scatter(d["hydro_year"], d["temp_mean"], s=18, color="#e67e22")
        ax.plot(d["hydro_year"], res.intercept + res.slope * d["hydro_year"], color="#7b241c")
        ax.set_title(f"{region}  ({res.slope * 10:+.2f} C/decade)")
    fig.suptitle("Annual mean temperature and linear trend", fontsize=14)
    fig.supylabel("Mean temperature (C)")
    fig.tight_layout()
    _finish(fig, "fig3_temp_trends.png", show)


def fig_hot_dry_share(annual, share, show=True):
    """Headline chart: share of dry years that were also hot, by period."""
    order = region_order(annual) + ["ALL REGIONS"]
    fig, ax = plt.subplots(figsize=(12, 5.5))
    share.loc[order].plot(kind="bar", color=["#5dade2", "#c0392b"], width=0.8, ax=ax)
    ax.set_ylabel("% of dry years that were also hot")
    ax.set_xlabel("")
    ax.set_title("Dry years now coincide with heat far more often")
    ax.grid(axis="x", visible=False)
    ax.legend(title="Period")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    _finish(fig, "fig4_hot_dry_share.png", show)


def fig_vulnerability(annual, show=True):
    """Mean rainfall vs variability; bubble size = number of hot-dry years."""
    g = annual.groupby("region")
    v = pd.DataFrame({"mean_rain": g["rain_mm"].mean(),
                      "cv": g["rain_mm"].std() / g["rain_mm"].mean(),
                      "hot_dry": g["hot_dry"].sum()})
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(v["mean_rain"], v["cv"], s=v["hot_dry"] * 60 + 40, color="#c0392b", alpha=0.6)
    for name, row in v.iterrows():
        ax.annotate(name, (row["mean_rain"], row["cv"]), xytext=(6, 6), textcoords="offset points")
    ax.set_xlabel("Mean annual rainfall (mm)")
    ax.set_ylabel("Rainfall variability (CV)")
    ax.set_title("Regional vulnerability\n(bubble size = number of hot-dry years)")
    fig.tight_layout()
    _finish(fig, "fig5_vulnerability.png", show)


def plot_all(annual, share, show=False):
    fig_anomaly_timeline(annual, show)
    fig_drought_heatmap(annual, show)
    fig_temperature_trends(annual, show)
    fig_hot_dry_share(annual, share, show)
    fig_vulnerability(annual, show)

## 2. Load and inspect the data
Downloads all 8 locations on the first run and caches them. `check_raw` verifies equal row counts, no duplicate dates, no missing flags, and that no two locations fell in the same NASA grid cell.

In [ ]:
from IPython.display import display

for path in (DATA_RAW, DATA_CLEAN, DATA_ANNUAL):
    path.parent.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if DATA_RAW.exists():
    raw = load_raw(DATA_RAW)
else:
    raw = fetch_all()
    raw.to_csv(DATA_RAW)
    raw = load_raw(DATA_RAW)

check_raw(raw)
display(raw.head())

## 3. Clean and prepare
Replace `-999` flags, fill short temperature gaps (never rainfall), add hydrological year, keep complete years only.

In [ ]:
df = clean(raw)
df.to_csv(DATA_CLEAN)
print(f"{df.shape[0]:,} daily rows, {df['hydro_year'].nunique()} hydrological years, {df['region'].nunique()} regions")
display(df.head())
display(df.describe().T)

## 4. Annual indicators and drought flags
One row per region and hydrological year: total rainfall, mean temperature, soil wetness, longest wet-season dry spell, and the drought flags.

In [ ]:
annual = add_flags(build_annual(df))
annual.to_csv(DATA_ANNUAL, index=False)
display(annual.head(10))

## 5. Analysis 1: trends (change over time)
Slope per decade for each region. `p_value < 0.05` counts as significant.

In [ ]:
trends = trend_table(annual)
print("Slope per decade:")
display(trends.pivot(index="region", columns="variable", values="slope_per_decade").round(3))
print("p-values:")
display(trends.pivot(index="region", columns="variable", values="p_value").round(3))

## 6. Analysis 2: comparison between periods
Means and counts for 1982-2003 vs 2004-2025, with percentage change in rainfall and temperature change.

In [ ]:
comparison = period_comparison(annual)
display(comparison.round(2))

## 7. Analysis 3: correlation and regional ranking
Rainfall vs temperature/soil wetness, and regions ranked by recent drought frequency and rainfall variability (CV).

In [ ]:
display(correlations(annual).round(2))
display(regional_ranking(annual).round(2))

## 8. Analysis 4: are dry years getting hotter?
Share of dry years that were also hot, per region and period, plus a pooled significance test.

In [ ]:
share, dry_counts, hot_dry_counts = hot_dry_share(annual)
display(share.round(0))
print(hot_dry_test(annual))

## 9. Visualisations

### 9.1 Rainfall anomaly timeline
Red bars are drought years (z <= -1).

In [ ]:
fig_anomaly_timeline(annual)

### 9.2 Drought heatmap
Dark red columns mark widespread drought years.

In [ ]:
fig_drought_heatmap(annual)

### 9.3 Temperature trends

In [ ]:
fig_temperature_trends(annual)

### 9.4 Headline: dry years and heat

In [ ]:
fig_hot_dry_share(annual, share)

### 9.5 Regional vulnerability

In [ ]:
fig_vulnerability(annual)

## 10. Findings
*Numbers below come from our run. Re-check them against the outputs above if the code or data change.*

1. **Warming is strong and universal:** +0.37 to +0.49 C per decade in all 8 regions (p < 0.001). The second period is 0.74-1.02 C warmer than the first.
2. **No decline in rainfall:** trends are positive in most regions and significant in Raqqa, Homs and Deir ez-Zor; the rest show no significant trend.
3. **Dry years did not become more frequent:** 60 dry region-years in 1982-2003 vs 48 in 2004-2025.
4. **But dry years now coincide with heat:** the share of dry years that were also hot rose from about 17% to 58% (pooled), and increased in every region.
5. **Most exposed regions:** Deir ez-Zor and Raqqa have the lowest rainfall and the highest variability.
6. **Validation:** the flagged drought years match known events (1989-90, 1999-2000, 2008, 2014, 2025).

## 11. Limitations
- NASA POWER is gridded reanalysis (about 50 km), not station data. Rainfall is model/satellite-derived and less reliable in arid areas; use it for trends and anomalies, not exact totals.
- Nearby places can share one grid cell, so results describe broad regional patterns, not individual cities.
- Only 44 years per region: small counts, and simple trend tests that ignore autocorrelation.
- Temperature z-scores use the full period, and all regions warmed, so the rise in hot-dry years partly reflects the warming trend itself. The honest claim is that dry years now occur on a warmer baseline.
- Pooled regions share droughts, so the pooled significance test is indicative only.
- No local water-use, irrigation or crop data.

## 12. Next steps
- Lagged drought-risk model (previous-year rainfall and temperature as predictors).
- Add soil-moisture and evaporation drought indices; compare with station data where available.
- Dashboard for decision support.